# Experiments Results

## Imports

In [49]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from tqdm.auto import tqdm

from swot_toolkit.analysis import open_sites_and_dates
from swot_toolkit.pipe2 import open_output_dir
from swot_toolkit.pipe4 import calc_swot_metrics, quality_flags_bad


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Read results

In [50]:
sites_dates = open_sites_and_dates("/data/swot/output")

In [ ]:
results = pd.DataFrame()


for region, dates in tqdm(sites_dates.items(), desc="Regions"):
    for date in dates:
        base_dir, aoi, _ = open_output_dir(region=region, date=date)

        metrics_df = pd.read_parquet(base_dir / f"results/{region}_{date}_metrics.parquet")
        metrics_df["Region"] = region
        metrics_df["Date"] = pd.to_datetime(date)
        results = pd.concat([results, metrics_df], axis=0)

columns = results.columns.to_list()[:-2]


Regions:   0%|          | 0/5 [00:00<?, ?it/s]

In [61]:
results

,A1B1,A1B2,A1B3,A1B4,A2B1,A2B2,A2B3,A2B4,A3B1,A3B2,A3B3,A3B4,A4B1,A4B2,A4B3,A4B4,Region,Date
iou,0.7136,0.7136,0.7136,0.7136,0.4373,0.7567,0.6072,0.7482,0.4373,0.7567,0.6072,0.7482,0.1200,0.1198,0.1189,0.1187,Curua-Una,2024-07-13
f1,0.8328,0.8328,0.8328,0.8328,0.6085,0.8615,0.7556,0.8560,0.6085,0.8615,0.7556,0.8560,0.2143,0.2139,0.2126,0.2122,Curua-Una,2024-07-13
precision,0.9091,0.9091,0.9091,0.9091,0.4710,0.8759,0.6869,0.8818,0.4710,0.8759,0.6869,0.8818,0.9911,0.9911,0.9910,0.9910,Curua-Una,2024-07-13
recall,0.7684,0.7684,0.7684,0.7684,0.8595,0.8476,0.8395,0.8316,0.8595,0.8476,0.8395,0.8316,0.1201,0.1199,0.1191,0.1188,Curua-Una,2024-07-13
iou,0.7525,0.7525,0.7525,0.7525,0.5833,0.8052,0.7163,0.7897,0.5833,0.8052,0.7163,0.7897,0.1277,0.1276,0.1277,0.1276,Curua-Una,2025-08-14
f1,0.8588,0.8588,0.8588,0.8588,0.7368,0.8921,0.8347,0.8825,0.7368,0.8921,0.8347,0.8825,0.2265,0.2264,0.2265,0.2264,Curua-Una,2025-08-14
precision,0.9343,0.9343,0.9343,0.9343,0.6456,0.9370,0.8451,0.9516,0.6456,0.9370,0.8451,0.9516,0.9719,0.9728,0.9719,0.9728,Curua-Una,2025-08-14
recall,0.7946,0.7946,0.7946,0.7946,0.8580,0.8512,0.8246,0.8227,0.8580,0.8512,0.8246,0.8227,0.1282,0.1281,0.1282,0.1281,Curua-Una,2025-08-14
iou,0.2894,0.2894,0.2894,0.2894,0.6291,0.6224,0.6291,0.6224,0.6291,0.6224,0.6291,0.6224,0.1832,0.1830,0.1832,0.1830,Northeast,2024-05-29
f1,0.4488,0.4488,0.4488,0.4488,0.7723,0.7673,0.7723,0.7673,0.7723,0.7673,0.7723,0.7673,0.3097,0.3094,0.3097,0.3094,Northeast,2024-05-29


## Pivot Results

In [ ]:
# Create MultiIndex columns from the A/B pattern
# First, parse the column names to extract A and B factors
a_factors = [col[0:2] for col in columns]  # e.g., 'A1', 'A2', 'A3', 'A4'
b_factors = [col[2:4] for col in columns]  # e.g., 'B1', 'B2', 'B3', 'B4'

pivoted_results = (
    results.copy().reset_index(names=["Metric"]).set_index(["Region", "Date", "Metric"])
)
pivoted_results.columns = pd.MultiIndex.from_arrays([a_factors, b_factors], names=["A", "B"])

pivoted_results = pivoted_results.stack(level="A", future_stack=True)
pivoted_results

B                                      B1      B2      B3      B4
Region    Date       Metric    A                                 
Curua-Una 2024-07-13 iou       A1  0.7136  0.7136  0.7136  0.7136
                               A2  0.4373  0.7567  0.6072  0.7482
                               A3  0.4373  0.7567  0.6072  0.7482
                               A4  0.1200  0.1198  0.1189  0.1187
                     f1        A1  0.8328  0.8328  0.8328  0.8328
...                                   ...     ...     ...     ...
Rio_Negro 2025-08-07 precision A4  0.9908  0.9918  0.9908  0.9918
                     recall    A1  0.8372  0.8372  0.8372  0.8372
                               A2  0.8906  0.8668  0.8743  0.8558
                               A3  0.7825  0.7805  0.7825  0.7805
                               A4  0.1484  0.1483  0.1484  0.1483

[160 rows x 4 columns]

## Appendix A - All Results

In [167]:
pivoted_results.loc["Rio_Branco"].reset_index().groupby(["A", "Metric"]).mean().drop("Date", axis=1).round(3)

B                B1     B2     B3     B4
A  Metric                               
A1 f1         0.566  0.566  0.566  0.566
   iou        0.410  0.410  0.410  0.410
   precision  0.623  0.623  0.623  0.623
   recall     0.520  0.520  0.520  0.520
A2 f1         0.424  0.568  0.484  0.554
   iou        0.272  0.412  0.322  0.396
   precision  0.318  0.609  0.421  0.614
   recall     0.637  0.533  0.569  0.506
A3 f1         0.423  0.545  0.454  0.535
   iou        0.271  0.394  0.299  0.381
   precision  0.354  0.586  0.412  0.588
   recall     0.548  0.510  0.508  0.491
A4 f1         0.217  0.216  0.217  0.216
   iou        0.139  0.137  0.139  0.137
   precision  0.484  0.484  0.484  0.484
   recall     0.140  0.139  0.140  0.139

In [168]:
pivoted_results.loc["Rio_Negro"].reset_index().groupby(["A", "Metric"]).mean().drop("Date", axis=1).round(3)

B                B1     B2     B3     B4
A  Metric                               
A1 f1         0.788  0.788  0.788  0.788
   iou        0.658  0.658  0.658  0.658
   precision  0.754  0.754  0.754  0.754
   recall     0.846  0.846  0.846  0.846
A2 f1         0.759  0.800  0.787  0.801
   iou        0.617  0.673  0.654  0.675
   precision  0.664  0.745  0.713  0.751
   recall     0.905  0.886  0.898  0.881
A3 f1         0.761  0.779  0.778  0.782
   iou        0.621  0.642  0.640  0.645
   precision  0.723  0.751  0.744  0.755
   recall     0.851  0.843  0.852  0.843
A4 f1         0.428  0.426  0.428  0.426
   iou        0.288  0.286  0.288  0.285
   precision  0.972  0.973  0.972  0.973
   recall     0.292  0.290  0.292  0.290

In [169]:
pivoted_results.loc["Rio_Madeira"].reset_index().groupby(["A", "Metric"]).mean().drop("Date", axis=1).round(3)

B                B1     B2     B3     B4
A  Metric                               
A1 f1         0.865  0.865  0.865  0.865
   iou        0.763  0.763  0.763  0.763
   precision  0.875  0.875  0.875  0.875
   recall     0.857  0.857  0.857  0.857
A2 f1         0.687  0.894  0.810  0.893
   iou        0.523  0.809  0.681  0.808
   precision  0.548  0.879  0.729  0.882
   recall     0.920  0.911  0.912  0.906
A3 f1         0.687  0.894  0.810  0.893
   iou        0.523  0.809  0.681  0.808
   precision  0.548  0.879  0.729  0.882
   recall     0.920  0.911  0.912  0.906
A4 f1         0.689  0.689  0.689  0.689
   iou        0.534  0.533  0.534  0.533
   precision  0.961  0.961  0.961  0.961
   recall     0.542  0.541  0.542  0.541

In [ ]:
pivoted_results.loc["Curua-Una"].reset_index().groupby(["A", "Metric"]).mean().drop("Date", axis=1).round(3)

B                B1     B2     B3     B4
A  Metric                               
A1 f1         0.846  0.846  0.846  0.846
   iou        0.733  0.733  0.733  0.733
   precision  0.922  0.922  0.922  0.922
   recall     0.782  0.782  0.782  0.782
A2 f1         0.673  0.877  0.795  0.869
   iou        0.510  0.781  0.662  0.769
   precision  0.558  0.906  0.766  0.917
   recall     0.859  0.849  0.832  0.827
A3 f1         0.673  0.877  0.795  0.869
   iou        0.510  0.781  0.662  0.769
   precision  0.558  0.906  0.766  0.917
   recall     0.859  0.849  0.832  0.827
A4 f1         0.220  0.220  0.220  0.219
   iou        0.124  0.124  0.123  0.123
   precision  0.982  0.982  0.981  0.982
   recall     0.124  0.124  0.124  0.123

In [170]:
pivoted_results.loc["Northeast"].reset_index().groupby(["A", "Metric"]).mean().drop("Date", axis=1).round(3)

B                B1     B2     B3     B4
A  Metric                               
A1 f1         0.490  0.490  0.490  0.490
   iou        0.325  0.325  0.325  0.325
   precision  0.931  0.931  0.931  0.931
   recall     0.334  0.334  0.334  0.334
A2 f1         0.820  0.819  0.820  0.819
   iou        0.698  0.697  0.698  0.697
   precision  0.883  0.891  0.883  0.891
   recall     0.769  0.762  0.769  0.762
A3 f1         0.820  0.819  0.820  0.819
   iou        0.698  0.697  0.698  0.697
   precision  0.883  0.891  0.883  0.891
   recall     0.769  0.762  0.769  0.762
A4 f1         0.484  0.484  0.484  0.484
   iou        0.337  0.337  0.337  0.337
   precision  0.976  0.976  0.976  0.976
   recall     0.343  0.343  0.343  0.343

## Calculate Overall Median

In [83]:
# To calculate median over all regions and dates, we group by A and Metric
overall_results = pivoted_results.groupby(level=["A", "Metric"]).median().sort_index().round(3)
overall_results

B                B1     B2     B3     B4
A  Metric                               
A1 f1         0.774  0.774  0.774  0.774
   iou        0.635  0.635  0.635  0.635
   precision  0.909  0.909  0.909  0.909
   recall     0.782  0.782  0.782  0.782
A2 f1         0.690  0.866  0.787  0.863
   iou        0.527  0.764  0.649  0.759
   precision  0.551  0.881  0.729  0.884
   recall     0.859  0.849  0.844  0.839
A3 f1         0.690  0.849  0.787  0.847
   iou        0.527  0.738  0.649  0.734
   precision  0.551  0.881  0.729  0.884
   recall     0.853  0.847  0.832  0.827
A4 f1         0.372  0.370  0.372  0.370
   iou        0.230  0.229  0.230  0.229
   precision  0.970  0.971  0.970  0.971
   recall     0.232  0.230  0.232  0.230

## Calculate the Median by Region

In [100]:
# To calculate the median by region, let's focus on the best scenario only
best_experiment = pivoted_results.xs("A2", level="A")[["B2"]]

region_results = best_experiment.groupby(level=["Region", "Metric"]).median().sort_index().round(3)

region_results


B                         B2
Region      Metric          
Curua-Una   f1         0.877
            iou        0.781
            precision  0.906
            recall     0.849
Northeast   f1         0.819
            iou        0.697
            precision  0.891
            recall     0.762
Rio_Branco  f1         0.568
            iou        0.412
            precision  0.609
            recall     0.533
Rio_Madeira f1         0.894
            iou        0.809
            precision  0.879
            recall     0.911
Rio_Negro   f1         0.800
            iou        0.673
            precision  0.745
            recall     0.886

In [154]:
import plotly.graph_objects as go

rename_regions = {
    "Rio_Madeira": "Rio Madeira",
    "Curua-Una": "Curuá-Una",
    "Rio_Negro": "Rio Negro",
    "Rio_Branco": "Rio Branco",
}

# Prepare data - pivot so metrics are columns
plot_data = region_results.reset_index().pivot(index="Region", columns="Metric", values="B2")

# Sort regions by F1 score (descending)
plot_data = plot_data.sort_values("f1", ascending=False).rename(index=rename_regions)

# Define colors for each metric (colorblind-friendly palette)
colors = {
    "f1": "#0987E7",  # Blue
    "iou": "#0CC228",  # Orange
    "precision": "#967024",  # Green
    "recall": "#CC78BC",  # Purple
}

# Create grouped bar chart
fig = go.Figure()

for metric in ["f1", "iou", "precision", "recall"]:
    fig.add_trace(
        go.Bar(
            name=metric.upper() if metric == "iou" else metric.capitalize(),
            x=plot_data.index,
            y=plot_data[metric],
            marker_color=colors[metric],
            width=0.175,
        )
    )

# Publication-ready layout
fig.update_layout(
    xaxis_title="Regions of Interest",
    yaxis_title="Metric Score",
    barmode="group",
    plot_bgcolor="white",
    paper_bgcolor="white",
    font={"family": "Arial, sans-serif", "size": 20, "color": "black"},
    legend={
        "title": {"text": "Metric", "font": {"size": 20}},
        "x": 1.02,
        "y": 1,
        "xanchor": "left",
        "yanchor": "top",
        "bgcolor": "rgba(255, 255, 255, 0.8)",
        "bordercolor": "black",
        "borderwidth": 1,
    },
    width=1000,
    height=600,
    margin={"t": 50, "b": 80, "l": 80, "r": 150},
)

# Configure axes
fig.update_xaxes(
    showgrid=False,
    showline=True,
    linewidth=2,
    linecolor="black",
    mirror=True,
    ticks="outside",
    tickwidth=2,
    tickcolor="black",
    tickangle=-45,
)

fig.update_yaxes(
    range=[0, 1],
    showgrid=False,
    gridwidth=1,
    gridcolor="lightgray",
    showline=True,
    linewidth=2,
    linecolor="black",
    mirror=True,
    ticks="outside",
    tickwidth=2,
    tickcolor="black",
    tickmode="linear",
    tick0=0.0,
    dtick=0.1,
)

fig

## Results by Version

In [ ]:
version_results = best_experiment.reset_index()
version_results.columns.name = None

version_results["Date"] = pd.to_datetime(version_results["Date"])
version_results["Version"] = version_results["Date"].map(
    lambda d: "Version C" if d.year < 2025 else "Version D"
)

version_results = (
    version_results.drop(columns=["Region", "Date"])
    .groupby(["Version", "Metric"])
    .median()
    .sort_index()
    .round(3)
)
version_results


B2
Version   Metric          
Version C f1         0.767
          iou        0.622
          precision  0.876
          recall     0.848
Version D f1         0.875
          iou        0.778
          precision  0.890
          recall     0.851

In [158]:
import plotly.graph_objects as go

# Prepare data - pivot so metrics are columns
plot_data = version_results.reset_index().pivot(index="Version", columns="Metric", values="B2")


# Create grouped bar chart
fig = go.Figure()

for metric in ["f1", "iou", "precision", "recall"]:
    fig.add_trace(
        go.Bar(
            name=metric.upper() if metric == "iou" else metric.capitalize(),
            x=plot_data.index,
            y=plot_data[metric],
            marker_color=colors[metric],
            width=0.175,
        )
    )

# Publication-ready layout
fig.update_layout(
    xaxis_title="SWOT Product Version",
    yaxis_title="Metric Score",
    barmode="group",
    plot_bgcolor="white",
    paper_bgcolor="white",
    font={"family": "Arial, sans-serif", "size": 20, "color": "black"},
    legend={
        "title": {"text": "Metric", "font": {"size": 20}},
        "x": 1.02,
        "y": 1,
        "xanchor": "left",
        "yanchor": "top",
        "bgcolor": "rgba(255, 255, 255, 0.9)",
        "bordercolor": "black",
        "borderwidth": 2,
        "font": {"size": 14},
    },
    width=800,
    height=400,
    margin={"t": 60, "b": 100, "l": 90, "r": 180},
)

# Configure axes
fig.update_xaxes(
    title_font={"size": 18},
    showgrid=False,
    showline=True,
    linewidth=2,
    linecolor="black",
    mirror=True,
    ticks="outside",
    tickwidth=2,
    tickcolor="black",
    tickfont={"size": 16},
)

fig.update_yaxes(
    title_font={"size": 18},
    range=[0, 1.0],
    showgrid=False,
    gridwidth=1,
    gridcolor="lightgray",
    showline=True,
    linewidth=2,
    linecolor="black",
    mirror=True,
    ticks="outside",
    tickwidth=2,
    tickcolor="black",
    tickfont={"size": 14},
    tickmode="linear",
    tick0=0.0,
    dtick=0.1,
)

fig

## Pivot the columns

In [ ]:
# Create MultiIndex columns from the A/B pattern
# First, parse the column names to extract A and B factors
a_factors = [col[0:2] for col in columns]  # e.g., 'A1', 'A2', 'A3', 'A4'
b_factors = [col[2:4] for col in columns]  # e.g., 'B1', 'B2', 'B3', 'B4'

# Create MultiIndex columns
median_results.columns = pd.MultiIndex.from_arrays([a_factors, b_factors], names=["A", "B"])
median_results

A               A1                                  A2                    \
B               B1       B2       B3       B4       B1       B2       B3   
f1         0.77420  0.77420  0.77420  0.77420  0.68990  0.86600  0.78670   
iou        0.63535  0.63535  0.63535  0.63535  0.52660  0.76370  0.64865   
precision  0.90870  0.90870  0.90870  0.90870  0.55120  0.88105  0.72940   
recall     0.78150  0.78150  0.78150  0.78150  0.85875  0.84940  0.84395   

A                       A3                                  A4           \
B               B4      B1       B2       B3       B4       B1       B2   
f1         0.86325  0.6899  0.84925  0.78670  0.84665  0.37185  0.37020   
iou        0.75945  0.5266  0.73820  0.64865  0.73420  0.23020  0.22885   
precision  0.88400  0.5512  0.88105  0.72940  0.88400  0.96995  0.97055   
recall     0.83915  0.8532  0.84715  0.83205  0.82715  0.23155  0.23020   

A                            
B               B3       B4  
f1         0.37185  0.37020  
iou        0.23020  0.22885  
precision  0.96995  0.97055  
recall     0.23155  0.23020

In [ ]:
stacked_median = median_results.stack(level="A")

In [57]:
stacked_median.swaplevel().sort_index().round(3)

B                B1     B2     B3     B4
A                                       
A1 f1         0.774  0.774  0.774  0.774
   iou        0.635  0.635  0.635  0.635
   precision  0.909  0.909  0.909  0.909
   recall     0.782  0.782  0.782  0.782
A2 f1         0.690  0.866  0.787  0.863
   iou        0.527  0.764  0.649  0.759
   precision  0.551  0.881  0.729  0.884
   recall     0.859  0.849  0.844  0.839
A3 f1         0.690  0.849  0.787  0.847
   iou        0.527  0.738  0.649  0.734
   precision  0.551  0.881  0.729  0.884
   recall     0.853  0.847  0.832  0.827
A4 f1         0.372  0.370  0.372  0.370
   iou        0.230  0.229  0.230  0.229
   precision  0.970  0.971  0.970  0.971
   recall     0.232  0.230  0.232  0.230

In [69]:
pivoted_results.swaplevel()

B                                      B1      B2      B3      B4
Region    Date       A  Metric                                   
Curua-Una 2024-07-13 A1 iou        0.7136  0.7136  0.7136  0.7136
                     A2 iou        0.4373  0.7567  0.6072  0.7482
                     A3 iou        0.4373  0.7567  0.6072  0.7482
                     A4 iou        0.1200  0.1198  0.1189  0.1187
                     A1 f1         0.8328  0.8328  0.8328  0.8328
...                                   ...     ...     ...     ...
Rio_Negro 2025-08-07 A4 precision  0.9908  0.9918  0.9908  0.9918
                     A1 recall     0.8372  0.8372  0.8372  0.8372
                     A2 recall     0.8906  0.8668  0.8743  0.8558
                     A3 recall     0.7825  0.7805  0.7825  0.7805
                     A4 recall     0.1484  0.1483  0.1484  0.1483

[160 rows x 4 columns]

In [ ]:
pivoted_results = pivoted_results.stack(level="A")
pivoted_results = pivoted_results.swaplevel().sort_index()


In [58]:
results.head()

,A1B1,A1B2,A1B3,A1B4,A2B1,A2B2,A2B3,A2B4,A3B1,A3B2,A3B3,A3B4,A4B1,A4B2,A4B3,A4B4,Region,Date
iou,0.7136,0.7136,0.7136,0.7136,0.4373,0.7567,0.6072,0.7482,0.4373,0.7567,0.6072,0.7482,0.1200,0.1198,0.1189,0.1187,Curua-Una,2024-07-13
f1,0.8328,0.8328,0.8328,0.8328,0.6085,0.8615,0.7556,0.8560,0.6085,0.8615,0.7556,0.8560,0.2143,0.2139,0.2126,0.2122,Curua-Una,2024-07-13
precision,0.9091,0.9091,0.9091,0.9091,0.4710,0.8759,0.6869,0.8818,0.4710,0.8759,0.6869,0.8818,0.9911,0.9911,0.9910,0.9910,Curua-Una,2024-07-13
recall,0.7684,0.7684,0.7684,0.7684,0.8595,0.8476,0.8395,0.8316,0.8595,0.8476,0.8395,0.8316,0.1201,0.1199,0.1191,0.1188,Curua-Una,2024-07-13
iou,0.7525,0.7525,0.7525,0.7525,0.5833,0.8052,0.7163,0.7897,0.5833,0.8052,0.7163,0.7897,0.1277,0.1276,0.1277,0.1276,Curua-Una,2025-08-14


## Group by year (i.e., version)

In [35]:
results[columns].groupby([results.index, results["Date"].dt.year]).median()

A1B1    A1B2    A1B3    A1B4    A2B1    A2B2    A2B3  \
          Date                                                           
f1        2024  0.7050  0.7050  0.7050  0.7050  0.6876  0.7647  0.7559   
          2025  0.8509  0.8509  0.8509  0.8509  0.7371  0.8762  0.8338   
iou       2024  0.5444  0.5444  0.5444  0.5444  0.5239  0.6191  0.6076   
          2025  0.7404  0.7404  0.7404  0.7404  0.5837  0.7797  0.7149   
precision 2024  0.9123  0.9123  0.9123  0.9123  0.5506  0.8785  0.6892   
          2025  0.9112  0.9112  0.9112  0.9112  0.6484  0.8929  0.8385   
recall    2024  0.7639  0.7639  0.7639  0.7639  0.8544  0.8428  0.8369   
          2025  0.7915  0.7915  0.7915  0.7915  0.8540  0.8472  0.8415   

                  A2B4    A3B1    A3B2    A3B3    A3B4    A4B1    A4B2  \
          Date                                                           
f1        2024  0.7647  0.6876  0.7647  0.7559  0.7647  0.3097  0.3094   
          2025  0.8738  0.7371  0.8690  0.8338  0.8690  0.4340  0.4310   
iou       2024  0.6191  0.5239  0.6191  0.6076  0.6191  0.1832  0.1830   
          2025  0.7759  0.5837  0.7684  0.7149  0.7684  0.2772  0.2747   
precision 2024  0.8835  0.5506  0.8785  0.6892  0.8835  0.9911  0.9911   
          2025  0.8983  0.6484  0.9004  0.8468  0.9004  0.9683  0.9686   
recall    2024  0.8292  0.8544  0.8428  0.8369  0.8292  0.1834  0.1832   
          2025  0.8398  0.8415  0.8398  0.8211  0.8192  0.2797  0.2772   

                  A4B3    A4B4  
          Date                  
f1        2024  0.3097  0.3094  
          2025  0.4340  0.4310  
iou       2024  0.1832  0.1830  
          2025  0.2772  0.2747  
precision 2024  0.9910  0.9910  
          2025  0.9683  0.9686  
recall    2024  0.1834  0.1832  
          2025  0.2797  0.2772